In [1]:
import os
from pyspark.sql import SparkSession

# Variables (le notebook ne voit pas celles du terminal)
os.environ["LAB_BUCKET_NAME"] = os.environ["KUBERNETES_NAMESPACE"]
os.environ["POLARIS_URI"] = "http://polaris:8181"
os.environ["POLARIS_CATALOG"] = os.environ["LAB_BUCKET_NAME"]
os.environ["AWS_REGION"] = "us-east-1"

with open("/home/onyxia/.polaris-credentials.env") as f:
    for line in f:
        line = line.strip()
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value.strip('"')

# Choix du badge
POLARIS_PRINCIPAL = "dev"  # "dev" or "analyst"
POLARIS_CREDENTIAL = os.environ[f"POLARIS_{POLARIS_PRINCIPAL.upper()}_CREDENTIAL"]

BUCKET = os.environ["LAB_BUCKET_NAME"]
POLARIS_URI = os.environ["POLARIS_URI"]
POLARIS_CATALOG = os.environ["POLARIS_CATALOG"]
ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"

spark = (
    SparkSession.builder
    .appName("lab-polaris")
    .master("local[*]")
    # S3 access
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN", ""))
    .config("spark.hadoop.fs.s3a.endpoint.region", os.environ.get("AWS_REGION"))
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Hadoop catalog of the Spark lab
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", f"s3a://{BUCKET}/warehouse")
    # Polaris catalog (REST)
    .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.type", "rest")
    .config("spark.sql.catalog.polaris.uri", f"{POLARIS_URI}/api/catalog")
    .config("spark.sql.catalog.polaris.warehouse", POLARIS_CATALOG)
    .config("spark.sql.catalog.polaris.credential", POLARIS_CREDENTIAL)
    .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")
    .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Polaris principal:", POLARIS_PRINCIPAL)

:: loading settings :: url = jar:file:/usr/local/lib/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/onyxia/.ivy2.5.2/cache
The jars for the packages stored in: /home/onyxia/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d31c9719-e863-41a1-9a3f-08ee56d82e4c;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
:: resolution report :: resolve 526ms :: artifacts dl 43ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |

Polaris principal: dev


In [3]:
print(spark.sparkContext.getConf().get("spark.hadoop.fs.s3.impl", None))
spark.sql("SHOW NAMESPACES IN lab").show()
spark.sql("SHOW NAMESPACES IN polaris").show()

org.apache.hadoop.fs.s3a.S3AFileSystem
+---------+
|namespace|
+---------+
|   silver|
+---------+

+---------+
|namespace|
+---------+
+---------+



In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.silver")

spark.sql(f"""
    CALL polaris.system.register_table(
      table => 'silver.orders',
      metadata_file => 's3a://{BUCKET}/warehouse/silver/orders/metadata/v4.metadata.json'
    )
""").show()

spark.sql("SELECT count(*) AS orders FROM polaris.silver.orders").show()

+-------------------+-------------------+----------------------+
|current_snapshot_id|total_records_count|total_data_files_count|
+-------------------+-------------------+----------------------+
|1220184473976417285|               2833|                     3|
+-------------------+-------------------+----------------------+



+------+
|orders|
+------+
|  2833|
+------+



In [5]:
spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM polaris.silver.orders.snapshots
""").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|   orders|      false|
+---------+---------+-----------+

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|4219974134046780544|overwrite|2026-10-28 15:48:20.462|
|7719747662070971571|append   |2026-11-03 19:54:57    |
|1220184473976417285|append   |2026-11-03 19:59:47.577|
+-------------------+---------+-----------------------+



In [6]:
stg_users = spark.read.parquet(f"s3a://{BUCKET}/silver/spark/users.parquet")
stg_users.writeTo("polaris.silver.users").createOrReplace()

spark.sql("SELECT count(*) AS users FROM polaris.silver.users").show()
spark.table("polaris.silver.users").printSchema()

+-----+
|users|
+-----+
|   50|
+-----+

root
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [7]:
spark.sql("ALTER TABLE polaris.silver.users ADD COLUMN country STRING")
spark.sql("DESCRIBE polaris.silver.users").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|  user_id|   string|   NULL|
| username|   string|   NULL|
|     name|   string|   NULL|
|      sex|   string|   NULL|
|    email|   string|   NULL|
|birthdate|     date|   NULL|
|   street|   string|   NULL|
|     city|   string|   NULL|
|    state|   string|   NULL|
| zip_code|   string|   NULL|
|  country|   string|   NULL|
+---------+---------+-------+



In [2]:
spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("SELECT product, count(*) FROM polaris.silver.orders GROUP BY product").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|   orders|      false|
|   silver|    users|      false|
+---------+---------+-----------+



+---------+--------+
|  product|count(1)|
+---------+--------+
|    donut|     462|
|   cookie|     469|
|croissant|     491|
|    bread|     459|
|  brioche|     468|
|    drink|     484|
+---------+--------+



In [3]:
spark.sql("INSERT INTO polaris.silver.orders SELECT * FROM polaris.silver.orders LIMIT 1")

20:59:56.175 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) is aborting.
20:59:56.264 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) aborted.


Py4JJavaError: An error occurred while calling o86.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_TABLE_SNAPSHOT
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:345)
	at org.apache.iceberg.rest.ErrorHandlers$CommitErrorHandler.accept(ErrorHandlers.java:137)
	at org.apache.iceberg.rest.ErrorHandlers$CommitErrorHandler.accept(ErrorHandlers.java:119)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:242)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:347)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:299)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:112)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:150)
	at org.apache.iceberg.rest.RESTTableOperations.commit(RESTTableOperations.java:206)
	at

In [4]:
spark.sql("ALTER TABLE polaris.silver.users ADD COLUMN phone STRING")

Py4JJavaError: An error occurred while calling o86.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_TABLE_SCHEMA
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:345)
	at org.apache.iceberg.rest.ErrorHandlers$CommitErrorHandler.accept(ErrorHandlers.java:137)
	at org.apache.iceberg.rest.ErrorHandlers$CommitErrorHandler.accept(ErrorHandlers.java:119)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:242)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:347)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:299)
	at org.apache.iceberg.rest.BaseHTTPClient.post(BaseHTTPClient.java:112)
	at org.apache.iceberg.rest.RESTClient.post(RESTClient.java:150)
	at org.apache.iceberg.rest.RESTTableOperations.commit(RESTTableOperations.java:206)
	at o

In [5]:
for sql in [
    "INSERT INTO polaris.silver.orders SELECT * FROM polaris.silver.orders LIMIT 1",
    "ALTER TABLE polaris.silver.users ADD COLUMN phone STRING",
]:
    try:
        spark.sql(sql)
    except Exception as e:
        lines = [l for l in str(e).splitlines() if "Exception" in l or "authorized" in l]
        print(sql.split()[0], "->", lines[:2])
        print()
        

21:03:35.129 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) is aborting.
21:03:35.202 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) aborted.
INSERT -> [": org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_TABLE_SNAPSHOT", '\tSuppressed: org.apache.spark.util.Utils$OriginalTryStackTraceException: Full stacktrace of original doTryWithCallerStacktrace caller']

ALTER -> [": org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'analyst' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, catalog_reader]' is not authorized for op ADD_T

In [2]:
spark.sql("""
    SELECT snapshot_id, parent_id, operation, committed_at
    FROM polaris.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-------------------+---------+-----------------------+
|snapshot_id        |parent_id          |operation|committed_at           |
+-------------------+-------------------+---------+-----------------------+
|4219974134046780544|NULL               |overwrite|2026-10-28 15:48:20.462|
|7719747662070971571|4219974134046780544|append   |2026-11-03 19:54:57    |
|1220184473976417285|7719747662070971571|append   |2026-11-03 19:59:47.577|
+-------------------+-------------------+---------+-----------------------+



In [3]:
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM polaris.silver.users.snapshots
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|3270219024235150290|overwrite|2026-11-03 20:54:36.394|
+-------------------+---------+-----------------------+



In [4]:
spark.sql("ALTER TABLE polaris.silver.users DROP COLUMN country")
spark.sql("DESCRIBE polaris.silver.users").show()

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|  user_id|   string|   NULL|
| username|   string|   NULL|
|     name|   string|   NULL|
|      sex|   string|   NULL|
|    email|   string|   NULL|
|birthdate|     date|   NULL|
|   street|   string|   NULL|
|     city|   string|   NULL|
|    state|   string|   NULL|
| zip_code|   string|   NULL|
+---------+---------+-------+



In [5]:
spark.sql("""
    SELECT snapshot_id, operation, committed_at
    FROM polaris.silver.users.snapshots
""").show(truncate=False)

spark.sql("""
    SELECT timestamp, file, latest_snapshot_id, latest_schema_id
    FROM polaris.silver.users.metadata_log_entries
    ORDER BY timestamp
""").show(truncate=False)

+-------------------+---------+-----------------------+
|snapshot_id        |operation|committed_at           |
+-------------------+---------+-----------------------+
|3270219024235150290|overwrite|2026-11-03 20:54:36.394|
+-------------------+---------+-----------------------+

+-----------------------+--------------------------------------------------------------------------------------------------------------+-------------------+----------------+
|timestamp              |file                                                                                                          |latest_snapshot_id |latest_schema_id|
+-----------------------+--------------------------------------------------------------------------------------------------------------+-------------------+----------------+
|2026-11-03 20:54:37.51 |s3://user-k-sison-ece/warehouse/silver/users/metadata/00000-7b0e5e51-2544-47a7-8eda-6494c07006a5.metadata.json|3270219024235150290|0               |
|2026-11-03 20:56:53.09

In [6]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.gold")

spark.sql("""
    CREATE OR REPLACE TABLE polaris.gold.orders_by_product AS
    SELECT product, count(*) AS orders, sum(quantity) AS quantity
    FROM polaris.silver.orders
    GROUP BY product
""")

spark.sql("SHOW NAMESPACES IN polaris").show()
spark.sql("SELECT * FROM polaris.gold.orders_by_product ORDER BY product").show()

+---------+
|namespace|
+---------+
|     gold|
|   silver|
+---------+

+---------+------+--------+
|  product|orders|quantity|
+---------+------+--------+
|    bread|   459|    1425|
|  brioche|   468|    1424|
|   cookie|   469|    1373|
|croissant|   491|    1523|
|    donut|   462|    1390|
|    drink|   484|    1493|
+---------+------+--------+



In [2]:
def test(label, sql):
    try:
        rows = spark.sql(sql).collect()
        print(f"OK         {label}: {rows[:3]}")
    except Exception as e:
        msg = [l for l in str(e).splitlines() if "Forbidden" in l or "authorized" in l]
        op = msg[0].split("for op ")[-1] if msg else str(e).splitlines()[0][:150]
        print(f"FORBIDDEN  {label}: op {op}")

test("list tables in silver", "SHOW TABLES IN polaris.silver")
test("read silver.orders",    "SELECT count(*) AS orders FROM polaris.silver.orders")
test("read silver.users",     "SELECT count(*) AS users FROM polaris.silver.users")
test("list tables in gold",   "SHOW TABLES IN polaris.gold")
test("read gold table",       "SELECT * FROM polaris.gold.orders_by_product")
test("list namespaces",       "SHOW NAMESPACES IN polaris")
test("write silver.orders",   "INSERT INTO polaris.silver.orders SELECT * FROM polaris.silver.orders LIMIT 1")

OK         list tables in silver: [Row(namespace='silver', tableName='users', isTemporary=False), Row(namespace='silver', tableName='orders', isTemporary=False)]


OK         read silver.orders: [Row(orders=2833)]
OK         read silver.users: [Row(users=50)]
FORBIDDEN  list tables in gold: op LIST_TABLES
FORBIDDEN  read gold table: op LOAD_TABLE
FORBIDDEN  list namespaces: op LIST_NAMESPACES


21:15:01.365 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) is aborting.
21:15:01.454 [Thread-4] ERROR org.apache.spark.sql.execution.datasources.v2.AppendDataExec - Data source write support IcebergBatchWrite(table=polaris.silver.orders, format=PARQUET) aborted.
FORBIDDEN  write silver.orders: op ADD_TABLE_SNAPSHOT


In [2]:
spark.sql("DROP TABLE IF EXISTS polaris.gold.orders_by_product")
spark.sql("DROP TABLE IF EXISTS polaris.silver.users")
spark.sql("DROP NAMESPACE IF EXISTS polaris.gold")

spark.sql("SHOW NAMESPACES IN polaris").show()
spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("SELECT count(*) AS orders FROM polaris.silver.orders").show()

+---------+
|namespace|
+---------+
|   silver|
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|   orders|      false|
+---------+---------+-----------+



+------+
|orders|
+------+
|  2833|
+------+



In [3]:
spark.stop()